# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR\u2072 dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata as a single object
meta = dataset.metadata

print(f"Name: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs. All references are made by `@id` according to the dataset schema.

Below, we enumerate the dataset's record sets and associated fields. For each, we print the `@id` and human-readable name.

In [ ]:
# List all record sets, their @id, and their fields
all_record_sets = meta.record_sets

print("Available record sets in dataset:\n")

record_set_ids = []
for rs in all_record_sets:
    rs_id = rs.id
    rs_name = rs.name if hasattr(rs, 'name') else rs_id
    record_set_ids.append(rs_id)
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {rs_name}")
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for field in rs.fields:
            field_id = field.id
            field_name = field.name if hasattr(field, 'name') else field_id
            field_dtype = getattr(field, 'data_type', 'unknown')
            print(f"    - Field @id: {field_id} (name: {field_name}, type: {field_dtype})")
    print("")

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames using their `@id`.

Modify the variable `selected_record_set_id` below to select a record set you'd like to explore:


In [ ]:
# Choose the record set to extract by its @id
selected_record_set_id = record_set_ids[0]  # Change the index if you want a different record set
print(f"Extracting data for RecordSet @id: {selected_record_set_id}")

# You can also extract multiple record sets by providing their @id in the list below
extract_record_set_ids = [selected_record_set_id]

dataframes = {}
for rs_id in extract_record_set_ids:
    print(f"Loading records for record set {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records with columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's process and analyze data using only field and column `@id`s.

### Steps:
- Select a numeric field by its `@id` for basic statistics and outlier removal
- Normalize the numeric field
- Optionally group by a categorical field


In [ ]:
df = dataframes[selected_record_set_id]

# Show all available columns for reference
print("Columns in the DataFrame:", df.columns.tolist())

# Pick a numeric field (by @id) from the DataFrame columns
from IPython.display import display
numeric_field_id = None
for col in df.columns:
    # Example: try to pick a field related to abundance, count, molecular weight, etc.
    if col.lower().startswith('cr:') and (('count' in col.lower()) or ('abundance' in col.lower()) or ('weight' in col.lower()) or (df[col].dtype in [int, float])):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Fallback: select the first numeric-looking column
    for col in df.select_dtypes(include=['float', 'int']).columns:
        numeric_field_id = col
        break

print(f"Selected numeric field for EDA: {numeric_field_id}")

if numeric_field_id is not None:
    # Drop missing values for this analysis
    df_clean = df.dropna(subset=[numeric_field_id])

    # Filter (choose a threshold, for demo we use mean)
    threshold = df_clean[numeric_field_id].mean()
    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]

    print(f"Records where {numeric_field_id} > {round(threshold, 2)}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_zscore"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"First 5 records with normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a likely field (categorical)
    group_field_id = None
    for col in df.columns:
        if col.lower().startswith('cr:') and (('type' in col.lower()) or ('group' in col.lower()) or ('accession' in col.lower()) or ('sample' in col.lower())):
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by {group_field_id} (showing mean of {numeric_field_id} by group):")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        display(grouped.sort_values(ascending=False).head())
else:
    print("No numeric field suitable for EDA found in this record set.")

## 5. Visualization

Visualize distributions or relationships between numeric fields in the dataset. All field references are by `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df_clean[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df_clean[group_field_id], y=df_clean[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion

This notebook demonstrated how to load, explore, and visualize the FAIR² mass spectrometry dataset using `mlcroissant` and referenced all data elements by their `@id`.

- We loaded dataset metadata and contents.
- We programmatically inspected all available record sets and fields.
- Using field `@id`s, we performed basic data filtering, normalization, grouping, and plotting.

**Next steps:** consider applying statistical or machine learning models using the processed DataFrame. For further documentation, see [`mlcroissant`](https://pypi.org/project/mlcroissant/) and [MLCommons Croissant documentation](https://mlcommons.github.io/croissant/).
